In [10]:
import os
import numpy as np
import pandas as pd
from skimage import io, measure

mask_dir = "/home/kayllany.oliveira/remote-repos/CellViability/resultados/cellpose_sam/"
resultados_detalhado = []
resultados_resumido = []

# Ordena os arquivos para processar em ordem alfabética
for fname in sorted(os.listdir(mask_dir)):
    if fname.endswith("_mask_labels.png"):
        mask_path = os.path.join(mask_dir, fname)
        mask = io.imread(mask_path)

        # --- Ajuste principal ---
        # PNG do Cellpose já contém rótulos inteiros (1,2,3... = células)
        # Se tiver múltiplos canais (RGB/RGBA), usa só o primeiro
        if mask.ndim == 3:
            mask = mask[:, :, 0].astype(np.int32)
        else:
            mask = mask.astype(np.int32)

        # O Cellpose já gera labels únicos, não precisa relabel
        mask_labeled = mask

        # Extrai propriedades de cada célula
        props = measure.regionprops_table(
            mask_labeled,
            properties=[
                "label",
                "area",
                "perimeter",
                "eccentricity",
                "solidity",
                "orientation",
                "major_axis_length",
                "minor_axis_length"
            ]
        )

        # DataFrame detalhado
        df_props = pd.DataFrame(props)
        df_props["arquivo"] = fname
        df_props["total_objetos"] = mask_labeled.max()
        resultados_detalhado.append(df_props)

        # Resumo por imagem
        resumo = {
            "arquivo": fname,
            "total_objetos": mask_labeled.max(),
            "area_media": df_props["area"].mean() if not df_props.empty else 0,
            "area_min": df_props["area"].min() if not df_props.empty else 0,
            "area_max": df_props["area"].max() if not df_props.empty else 0,
            "perimeter_media": df_props["perimeter"].mean() if not df_props.empty else 0,
            "eccentricity_media": df_props["eccentricity"].mean() if not df_props.empty else 0,
            "solidity_media": df_props["solidity"].mean() if not df_props.empty else 0,
            "major_axis_media": df_props["major_axis_length"].mean() if not df_props.empty else 0,
            "minor_axis_media": df_props["minor_axis_length"].mean() if not df_props.empty else 0
        }
        resultados_resumido.append(resumo)

# Salva CSV detalhado
df_detalhado = pd.concat(resultados_detalhado, ignore_index=True)
csv_detalhado_path = os.path.join(mask_dir, "detalhado_CP_4.csv")
df_detalhado.to_csv(csv_detalhado_path, index=False)

# Salva CSV resumido
df_resumido = pd.DataFrame(resultados_resumido)
csv_resumido_path = os.path.join(mask_dir, "resumo_CP_4.csv")
df_resumido.to_csv(csv_resumido_path, index=False)

print(f"Análise concluída!")
print(f"CSV detalhado salvo em: {csv_detalhado_path}")
print(f"CSV resumido salvo em: {csv_resumido_path}")


Análise concluída!
CSV detalhado salvo em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/cellpose_sam/detalhado_CP_4.csv
CSV resumido salvo em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/cellpose_sam/resumo_CP_4.csv
